In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def multistage_interpolation_simulation(L1, L2):
    clear_output(wait=True)
    
    L_total = L1 * L2
    
    # 1. Frequency vector covering [-1.5*pi, 1.5*pi]
    N_fft = 4000
    w = np.linspace(-1.5 * np.pi, 1.5 * np.pi, N_fft)
    
    # Base Signal Spectrum X(e^{j\omega})
    omega_N = 0.3 * np.pi
    
    def base_spectrum(w_axis):
        spec = np.zeros_like(w_axis)
        mask = np.abs(w_axis) <= omega_N
        spec[mask] = 1.0 - np.abs(w_axis[mask]) / omega_N
        return spec

    # Original Spectrum
    x_spec = base_spectrum(w)
    
    # --- STAGE 1: Interpolation by L1 (Expansion by L1 followed by low-pass filtering) ---
    # Expansion compresses the frequency axis by L1 and scales amplitude by L1
    x_expanded_1 = np.zeros_like(w)
    for k in range(-5, 6):
        x_expanded_1 += L1 * np.interp(w * L1 - 2.0 * np.pi * k, w, x_spec, left=0, right=0)
        
    # Low-pass filter for Stage 1 with cutoff pi / L1
    wc1 = np.pi / L1
    h1_spec = np.where(np.abs(w) <= wc1, 1.0, 0.0)
    y1_spec = x_expanded_1 * h1_spec
    
    # --- STAGE 2: Interpolation by L2 (applied to Stage 1 output) ---
    x_expanded_2 = np.zeros_like(w)
    for k in range(-5, 6):
        x_expanded_2 += L2 * np.interp(w * L2 - 2.0 * np.pi * k, w, y1_spec, left=0, right=0)
        
    # Low-pass filter for Stage 2 with cutoff pi / L2
    wc2 = np.pi / L2
    h2_spec = np.where(np.abs(w) <= wc2, 1.0, 0.0)
    y2_spec = x_expanded_2 * h2_spec

    # --- SINGLE-STAGE COMPARISON (Interpolation by L_total) ---
    x_expanded_total = np.zeros_like(w)
    for k in range(-5, 6):
        x_expanded_total += L_total * np.interp(w * L_total - 2.0 * np.pi * k, w, x_spec, left=0, right=0)
        
    wc_total = np.pi / L_total
    h_total_spec = np.where(np.abs(w) <= wc_total, 1.0, 0.0)
    y_total_spec = x_expanded_total * h_total_spec

    # --- Plotting ---
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # Subplot 1: Stage 1
    axes[0].plot(w / np.pi, x_spec, color='tab:blue', lw=2, label=r'Original Spectrum $\mathcal{X}(e^{j\omega})$')
    axes[0].plot(w / np.pi, y1_spec, color='tab:orange', lw=2.5, linestyle='-', label=r'After Stage 1 ($L_1=' + str(L1) + '$)')
    axes[0].axvline(x=wc1 / np.pi, color='green', linestyle=':', lw=1.5, label=r'Cutoff $\pi/L_1$')
    axes[0].axvline(x=-wc1 / np.pi, color='green', linestyle=':', lw=1.5)
    axes[0].set_title(r'Stage 1: Oversampling by $L_1 = ' + str(L1) + '$', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_xlim(-1.5, 1.5)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Subplot 2: Stage 2
    axes[1].plot(w / np.pi, y1_spec, color='tab:orange', lw=1.5, linestyle='--', label=r'Input to Stage 2 ($\mathcal{Y}_1$)')
    axes[1].plot(w / np.pi, y2_spec, color='tab:purple', lw=2.5, label=r'Final Multi-Stage Output ($L = L_1 \times L_2 = ' + str(L_total) + '$)')
    axes[1].axvline(x=wc2 / np.pi, color='red', linestyle=':', lw=1.5, label=r'Cutoff $\pi/L_2$')
    axes[1].axvline(x=-wc2 / np.pi, color='red', linestyle=':', lw=1.5)
    axes[1].set_title(r'Stage 2: Oversampling by $L_2 = ' + str(L2) + '$ (Total Factor $L = ' + str(L_total) + '$)', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=9)
    axes[1].set_xlim(-1.5, 1.5)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Subplot 3: Equivalence Verification
    axes[2].plot(w / np.pi, y2_spec, color='tab:purple', lw=2.5, label=r'Multi-Stage Result ($L_1 \times L_2$)')
    axes[2].plot(w / np.pi, y_total_spec, color='black', lw=1.5, linestyle='--', label=r'Single-Stage Equivalent ($L=' + str(L_total) + '$)')
    axes[2].set_title(r'Equivalence Verification: Multi-Stage vs Single-Stage ($L=' + str(L_total) + '$)', fontsize=10, fontweight='bold')
    axes[2].set_xlabel(r'Normalized Frequency ($\omega / \pi$)', fontsize=9)
    axes[2].set_ylabel('Amplitude', fontsize=9)
    axes[2].set_xlim(-1.5, 1.5)
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Conclusion Box displayed under the plots
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 12px; border-radius: 5px; background-color: #f9f9f9; font-family: sans-serif; font-size: 14px;">
        <strong>Conclusion:</strong> 
        The plots above demonstrate the equivalence between a <strong>multi-stage oversampling (interpolation) system</strong> (composed of two sequential stages with factors $L_1 = {L1}$ and $L_2 = {L2}$) and a <strong>single-stage oversampling system</strong> with a total factor of $L = {L_total}$. 
        As verified in the third subplot, both approaches yield identical final spectra, confirming that breaking down the interpolation process into multiple stages achieves significant computational and hardware efficiency without altering the theoretical outcome.
    </div>
    """))

# Interactive Sliders for L1 and L2
l1_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description='Stage 1 ($L_1$):', style={'description_width': 'initial'})
l2_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description='Stage 2 ($L_2$):', style={'description_width': 'initial'})

ui = widgets.VBox([l1_slider, l2_slider])
display(ui)

out = widgets.interactive_output(multistage_interpolation_simulation, {'L1': l1_slider, 'L2': l2_slider})
display(out)